# 🧬 Regime Lab – Quant Researcher
**Workflow**: GMM Regime Detection → Regime-Conditional Analytics → Interactive Timeline Visualisation

---

In [ ]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

session = AnalystSession(role=RoleContext.RESEARCHER)
SYMBOL = 'BTC-USD'
TIMEFRAME = '1d'
N_REGIMES = 3

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Feature Engineering for Clustering
Extracting volatility and momentum signatures for market segmentation.

In [ ]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME)
except Exception:
    df = session.sample_ohlcv(symbol='BTC', days=730)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 21])
df = df.drop_nulls()

regime_features = ['returns', 'vol_21', 'rsi_14']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.select(regime_features).to_numpy())
print(f"Scaled features shape: {X_scaled.shape}")

## 2. GMM Regime Identification
Clustering market states using Gaussian Mixture Models.

In [ ]:
gmm = GaussianMixture(n_components=N_REGIMES, covariance_type='full', random_state=42)
gmm.fit(X_scaled)
labels = gmm.predict(X_scaled)
probs  = gmm.predict_proba(X_scaled)

df = df.with_columns(pl.Series('regime', labels))
print(f" BIC: {gmm.bic(X_scaled):.2f}")

## 3. Interactive Regime Timeline
Overlaying detected regimes on the asset price curve.

In [ ]:
palette = ['#34d399', '#f87171', '#a78bfa', '#fbbf24']
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['close'], name='Price', line=dict(color='white', width=1)))

for r in range(N_REGIMES):
    r_df = df.filter(pl.col('regime') == r)
    fig.add_trace(go.Scatter(
        x=r_df['timestamp'], 
        y=r_df['close'], 
        mode='markers', 
        name=f'Regime {r}', 
        marker=dict(color=palette[r % len(palette)], size=4)
    ))

fig.update_layout(
    title=f"{SYMBOL} – Market Regime Clustering",
    yaxis_title="Price",
    height=500, 
    **PLOTLY_DARK
)
fig.show()

## 4. Conditional Statistics
Validating if regimes represent distinct statistical states.

In [ ]:
stats = df.group_by('regime').agg([
    pl.col('returns').mean().alias('mu'),
    pl.col('returns').std().alias('sigma'),
    pl.len().alias('count')
]).sort('regime')

fig_bar = px.bar(stats.to_pandas(), x='regime', y='mu', color='sigma', 
                 title="Regime-Conditional Returns & Volatility", 
                 color_continuous_scale="Viridis")
fig_bar.update_layout(**PLOTLY_DARK)
fig_bar.show()